In [ ]:
# pip install pandas openpyxl

In [ ]:
import pandas as pd
import os
import glob
from pathlib import Path

In [ ]:
# ====================== 配置项（请修改为实际路径） ======================
# 初始数据根文件夹
BASE_DIR = r"C:\Users\33759\Desktop\陈正扬\初始数据"
# 不含学科数据输出文件夹
OUTPUT_DIR_WITHOUT_SUBJECT = r"C:\Users\33759\Desktop\陈正扬\处理后数据（不含学科）"

# 年份范围
YEARS = ["2020", "2021", "2022", "2023", "2024", "2025"]
# 需要忽略/特殊处理的文件名
IGNORE_FILES = ["高校辅导员研究.xlsx", "中国特色社会主义理论体系.xlsx"]
# 专项项目映射
SPECIAL_PROJECT_MAP = {
    "高校辅导员研究.xlsx": "专项项目（辅导员研究）",
    "中国特色社会主义理论体系.xlsx": "专项项目（中国特色社会主义）"
}

# ====================== 通用工具函数 ======================
def read_excel_auto_header(file_path):
    # 先读取所有行（不设表头）
    df_raw = pd.read_excel(file_path, header=None)
    header_row = None

    # 遍历每行，查找包含核心表头的行
    for idx, row in df_raw.iterrows():
        # 清理行内容，统一小写，便于匹配
        row_clean = " ".join([str(cell).strip().lower() for cell in row if pd.notna(cell)])
        if "项目名称" in row_clean and "学校名称" in row_clean:
            header_row = idx
            break

    if header_row is None:
        raise ValueError(f"文件 {file_path} 未找到包含'项目名称'和'学校名称'的表头行！")

    # 重新读取，设置正确表头
    df = pd.read_excel(file_path, header=header_row)
    # 清理列名（去空格、特殊字符）
    df.columns = [str(col).strip() for col in df.columns]
    return df

def clean_dataframe(df):
    # 1. 删除完全重复的行
    df = df.drop_duplicates()
    # 2. 删除核心列（项目名称/学校名称）缺失的行
    core_cols = ["项目名称", "学校名称"]
    df = df.dropna(subset=core_cols, how="any")
    # 3. 其他列缺失值填充为空字符串（避免后续处理报错）
    df = df.fillna("")
    return df

def unify_column_names(df):
    col_mapping = {"学科": "学科门类"}
    df = df.rename(columns=col_mapping)
    return df

# ====================== 处理不含学科的数据集 ======================
def process_data_without_subject():
    print("\n=== 开始处理【不含学科】数据集 ===")
    all_dfs = []

    # 遍历年份文件夹
    for year in YEARS:
        year_dir = os.path.join(BASE_DIR, year)
        if not os.path.exists(year_dir):
            print(f"警告：年份文件夹 {year_dir} 不存在，跳过")
            continue

        # 遍历该年份下所有Excel文件
        excel_files = glob.glob(os.path.join(year_dir, "*.xlsx"))
        for file_path in excel_files:
            file_name = os.path.basename(file_path)

            try:
                # 读取文件（自动识别表头）
                df = read_excel_auto_header(file_path)
                # 统一列名
                df = unify_column_names(df)

                # 处理专项项目：添加项目类别列
                if file_name in SPECIAL_PROJECT_MAP:
                    df["项目类别"] = SPECIAL_PROJECT_MAP[file_name]

                # 删除学科相关列
                subject_cols = ["学科门类", "学科"]
                for col in subject_cols:
                    if col in df.columns:
                        df = df.drop(columns=[col])

                # 添加立项年份列
                df["立项年份"] = year
                # 加入列表
                all_dfs.append(df)
                print(f"成功处理：{file_path}")
            except Exception as e:
                print(f"处理失败 {file_path}：{str(e)}")

    # 合并并清洗
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        combined_df = clean_dataframe(combined_df)
        # 保存
        output_path = os.path.join(OUTPUT_DIR_WITHOUT_SUBJECT, "合并数据（不含学科）.xlsx")
        combined_df.to_excel(output_path, index=False)
        print(f"【不含学科】数据集保存完成：{output_path}")
    else:
        print("警告：无【不含学科】数据可处理！")

# ====================== 主执行逻辑 ======================
if __name__ == "__main__":
    # 确保输出文件夹存在
    Path(OUTPUT_DIR_WITHOUT_SUBJECT).mkdir(parents=True, exist_ok=True)

    # 执行处理
    process_data_without_subject()

    print("\n=== 所有数据处理完成！ ===")